# Lab 5: Workflow Design with LLMs

### Exercise 1 — Customer Review to Support Ticket Pipeline

Build a 2-step pipeline. Step 1 extracts the core complaint, product/feature mentioned, and customer sentiment from a raw customer review as structured data. Step 2 generates a support ticket summary for the internal team, using only the structured data from Step 1, not the raw review text.

**Input:** `review_text`

**Output:** the structured complaint data from Step 1, and the support ticket summary from Step 2

**Step 1 — Template:** `PromptTemplate` | **Parser:** `PydanticOutputParser` (fields: `complaint`, `product_or_feature`, `sentiment`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "support ticket writer") | **Parser:** `StrOutputParser`

In [40]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field

In [41]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"]
)

In [43]:
review_text = """
I purchased the AeroSound wireless headphones last week.
The sound quality is excellent, but the battery drains within
two hours and the headphones disconnect frequently.
I would like help resolving these issues.
"""

input_temp = PromptTemplate.from_template(
    template="The customer review regarding the product is {review_text}"
)

input = input_temp.invoke({"review_text": review_text})

In [44]:
class CustomerReview(BaseModel):
    complaint: str = Field(description="The complaint of the customer")
    product_or_feature: str = Field(description="The product or feature mentioned in the review")
    sentiment: str = Field(description="The sentiment of the customer's review")

In [45]:
pydantic_parser = PydanticOutputParser(pydantic_object=CustomerReview)

In [46]:
input_temp = PromptTemplate.from_template(
    template="""The customer review is {review_text}.
    Extract the complaint, product or feature, and sentiment.
    Return the result in the required JSON format.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

input = input_temp.invoke({"review_text": review_text})

response = llm.invoke(input)

structured_data = pydantic_parser.parse(response.content)

print(structured_data)

complaint='Battery drains within two hours and the headphones disconnect frequently.' product_or_feature='AeroSound wireless headphones' sentiment='negative'


In [47]:
ticket_temp = ChatPromptTemplate([
    ("system", "You are a support ticket writer who prepares concise and clear support tickets for an internal customer support team."),
    ("user", """Prepare a support ticket using only the following structured customer review data.

    Complaint: {complaint}
    Product or Feature: {product_or_feature}
    Sentiment: {sentiment}

    Include the issue, product or feature, customer sentiment, and requested action.""")
])

In [48]:
input = ticket_temp.invoke({
    "complaint": structured_data.complaint,
    "product_or_feature": structured_data.product_or_feature,
    "sentiment": structured_data.sentiment
})

response = llm.invoke(input)

str_parser = StrOutputParser()

support_ticket = str_parser.parse(response.content)

print(support_ticket)

**Support Ticket**

- **Issue:** Battery drains within two hours and the headphones disconnect frequently.  
- **Product/Feature:** AeroSound wireless headphones  
- **Customer Sentiment:** Negative  
- **Requested Action:** Investigate the battery and connectivity problems and provide a resolution (e.g., repair, replacement, or refund) for the affected unit.


---

### Exercise 2 — Research Paper Abstract to Layperson Summary Pipeline

Build a 2-step pipeline. Step 1 extracts the research question, method, and key finding from a paper abstract as structured data. Step 2 generates a plain-language summary for a non-expert audience, using only the structured version from Step 1, not the original abstract.

**Input:** `abstract_text`

**Output:** the structured extraction from Step 1, and the layperson summary from Step 2

**Step 1 — Template:** `PromptTemplate` | **Parser:** `PydanticOutputParser` (fields: `research_question`, `method`, `key_finding`)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "science communicator explaining to a general audience") | **Parser:** `StrOutputParser`

In [49]:
abstract_text = """
This study investigates whether regular physical activity improves sleep
quality among university students. Researchers conducted a 12-week study
involving 240 students who were divided into an exercise group and a control
group. Sleep quality was measured using standardized questionnaires before
and after the study. The results showed that students who participated in
regular physical activity reported better sleep quality than those in the
control group. The findings suggest that regular exercise may contribute
to improved sleep among university students.
"""

In [50]:
class ResearchAbstract(BaseModel):
    research_question: str = Field(description="The main research question")
    method: str = Field(description="The research method used in the study")
    key_finding: str = Field(description="The main finding of the research")

In [51]:
pydantic_parser = PydanticOutputParser(pydantic_object=ResearchAbstract)

In [52]:
input_temp = PromptTemplate.from_template(
    template="""Extract the research question, method, and key finding
    from the following research paper abstract.

    Abstract: {abstract_text}

    Return the result in the required JSON format.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

In [53]:
input = input_temp.invoke({"abstract_text": abstract_text})

response = llm.invoke(input)

structured_data = pydantic_parser.parse(response.content)

print(structured_data)

research_question='Does regular physical activity improve sleep quality among university students?' method='A 12-week experimental study with 240 university students randomly assigned to an exercise group or a control group; sleep quality was assessed using standardized questionnaires before and after the intervention.' key_finding='Students who participated in regular physical activity reported significantly better sleep quality than those in the control group, suggesting that regular exercise may improve sleep among university students.'


In [54]:
summary_temp = ChatPromptTemplate([
    ("system", "You are a science communicator explaining research findings to a general audience using simple, everyday language."),
    ("user", """Explain the following research in simple language for someone
    without a scientific background.

    Research Question: {research_question}
    Method: {method}
    Key Finding: {key_finding}

    Write a short, clear summary without adding unsupported claims.""")
])

In [55]:
input = summary_temp.invoke({
    "research_question": structured_data.research_question,
    "method": structured_data.method,
    "key_finding": structured_data.key_finding
})

response = llm.invoke(input)

str_parser = StrOutputParser()

summary = str_parser.parse(response.content)

print(summary)

**Simple Summary**

Researchers wanted to know if exercising regularly helps university students sleep better.  

- **How they tested it:** 240 students were split at random into two groups. One group did a regular exercise program for 12 weeks, while the other group did not change anything. Before the program started and after it ended, all students filled out the same well‑known sleep questionnaire that measures how good their sleep is.  

- **What they found:** After 12 weeks, the students who exercised reported noticeably better sleep than the students who didn’t exercise.  

**Bottom line:** For university students, doing regular physical activity can lead to improved sleep quality. No other benefits or long‑term effects were claimed in this study.


---

### Exercise 3 — Interview Transcript to Candidate Scorecard Pipeline

Build a 3-step pipeline. Step 1 extracts the candidate's answers to each interview question from a raw transcript. Step 2 evaluates each answer against a given set of skills, flagging any skill that wasn't clearly demonstrated. Step 3 formats the evaluation into a structured hiring scorecard with an overall recommendation.

**Input:** `transcript_text`, `skills_to_assess`

**Output:** the extracted answers from Step 1, the flagged skill evaluation from Step 2, and the final scorecard from Step 3

**Step 1 — Template:** `PromptTemplate` | **Parser:** `PydanticOutputParser` (field: list of `question` + `answer` pairs)

**Step 2 — Template:** `ChatPromptTemplate` (system persona: "technical interviewer evaluating skill evidence") | **Parser:** `PydanticOutputParser` (fields: `skill`, `demonstrated: bool`, `evidence_or_gap`)

**Step 3 — Template:** `ChatPromptTemplate` (system persona: "hiring panel coordinator") | **Parser:** `PydanticOutputParser` (fields: list of scorecard rows + `overall_recommendation`)


In [56]:
transcript_text = """
Interviewer: Can you explain how you would troubleshoot a web application
that returns a 500 Internal Server Error?

Candidate: I would check the application logs, reproduce the issue,
inspect recent code changes, and verify the database connection.

Interviewer: How have you used Git in a team project?

Candidate: I have used Git to commit changes, create branches, and merge
my work after reviewing the team's pull requests.

Interviewer: How would you deploy an application to a cloud platform?

Candidate: I have not deployed an application to a cloud platform yet,
but I have studied the basic deployment process.
"""

skills_to_assess = [
    "Troubleshooting",
    "Git collaboration",
    "Cloud deployment"
]

In [60]:
class InterviewAnswer(BaseModel):
    question: str = Field(description="The interview question asked")
    answer: str = Field(description="The candidate's answer")


class ExtractedAnswers(BaseModel):
    answers: list[InterviewAnswer] = Field(
        description="List of interview questions and candidate answers"
    )

In [61]:
pydantic_parser = PydanticOutputParser(pydantic_object=ExtractedAnswers)

In [62]:
input_temp = PromptTemplate.from_template(
    template="""Extract each interview question and the candidate's answer
    from the following transcript.

    Transcript: {transcript_text}

    Return the result in the required JSON format.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

In [63]:
input = input_temp.invoke({"transcript_text": transcript_text})

response = llm.invoke(input)

extracted_data = pydantic_parser.parse(response.content)

print(extracted_data)

answers=[InterviewAnswer(question='Can you explain how you would troubleshoot a web application that returns a 500 Internal Server Error?', answer='I would check the application logs, reproduce the issue, inspect recent code changes, and verify the database connection.'), InterviewAnswer(question='How have you used Git in a team project?', answer="I have used Git to commit changes, create branches, and merge my work after reviewing the team's pull requests."), InterviewAnswer(question='How would you deploy an application to a cloud platform?', answer='I have not deployed an application to a cloud platform yet, but I have studied the basic deployment process.')]


In [64]:
class SkillEvaluation(BaseModel):
    skill: str = Field(description="The skill being evaluated")
    demonstrated: bool = Field(description="Whether the candidate clearly demonstrated the skill")
    evidence_or_gap: str = Field(description="Evidence from the answers or the gap in demonstrated skill")


class SkillEvaluations(BaseModel):
    evaluations: list[SkillEvaluation] = Field(
        description="Evaluation of each skill being assessed"
    )

In [65]:
evaluation_parser = PydanticOutputParser(pydantic_object=SkillEvaluations)

In [66]:
evaluation_temp = ChatPromptTemplate([
    ("system", "You are a technical interviewer evaluating skill evidence. Evaluate only the evidence in the candidate's answers. Do not assume skills that were not demonstrated."),
    ("user", """Evaluate the candidate's answers against these skills:
    {skills_to_assess}

    Extracted interview answers:
    {extracted_answers}

    Mark each skill as demonstrated or not demonstrated, and explain the
    evidence or gap. Return the result in the required JSON format.
    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=evaluation_parser.get_format_instructions())

In [67]:
input = evaluation_temp.invoke({
    "skills_to_assess": skills_to_assess,
    "extracted_answers": extracted_data.model_dump_json()
})

response = llm.invoke(input)

evaluation_data = evaluation_parser.parse(response.content)

print(evaluation_data)

evaluations=[SkillEvaluation(skill='Troubleshooting', demonstrated=True, evidence_or_gap='The candidate described a systematic approach: checking application logs, reproducing the issue, inspecting recent code changes, and verifying the database connection, which clearly demonstrates troubleshooting ability.'), SkillEvaluation(skill='Git collaboration', demonstrated=True, evidence_or_gap="The candidate explained using Git to commit changes, create branches, and merge work after reviewing the team's pull requests, showing practical experience with Git in a collaborative setting."), SkillEvaluation(skill='Cloud deployment', demonstrated=False, evidence_or_gap='The candidate has not actually deployed an application to a cloud platform; they only mention having studied the basic deployment process, indicating a lack of hands‑on experience.')]


In [68]:
class ScorecardRow(BaseModel):
    skill: str = Field(description="The skill assessed")
    demonstrated: bool = Field(description="Whether the skill was demonstrated")
    evidence_or_gap: str = Field(description="Evidence supporting the evaluation")


class CandidateScorecard(BaseModel):
    scorecard: list[ScorecardRow] = Field(
        description="Structured scorecard for the candidate"
    )
    overall_recommendation: str = Field(
        description="Overall hiring recommendation based on the available evidence"
    )

In [69]:
scorecard_parser = PydanticOutputParser(pydantic_object=CandidateScorecard)

In [70]:
scorecard_temp = ChatPromptTemplate([
    ("system", "You are a hiring panel coordinator preparing a structured candidate scorecard based on the technical interview evaluation."),
    ("user", """Prepare a hiring scorecard using only the skill evaluation below.

    Skill Evaluation:
    {evaluation_data}

    Include every evaluated skill, its demonstrated status, and its evidence
    or gap. Provide an overall recommendation based on the available evidence.
    Return the result in the required JSON format.
    {JSON_FORMATTING}""")
]).partial(JSON_FORMATTING=scorecard_parser.get_format_instructions())

In [71]:
input = scorecard_temp.invoke({
    "evaluation_data": evaluation_data.model_dump_json()
})

response = llm.invoke(input)

scorecard_data = scorecard_parser.parse(response.content)

print(scorecard_data)

scorecard=[ScorecardRow(skill='Troubleshooting', demonstrated=True, evidence_or_gap='The candidate described a systematic approach: checking application logs, reproducing the issue, inspecting recent code changes, and verifying the database connection, which clearly demonstrates troubleshooting ability.'), ScorecardRow(skill='Git collaboration', demonstrated=True, evidence_or_gap="The candidate explained using Git to commit changes, create branches, and merge work after reviewing the team's pull requests, showing practical experience with Git in a collaborative setting."), ScorecardRow(skill='Cloud deployment', demonstrated=False, evidence_or_gap='The candidate has not actually deployed an application to a cloud platform; they only mention having studied the basic deployment process, indicating a lack of hands‑on experience.')] overall_recommendation='Strong in troubleshooting and Git collaboration, but lacks hands‑on cloud deployment experience. Recommend moving forward for roles focu

---

### Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline

Build a 3-step pipeline. Step 1 extracts the event details (event type, planned date, location, indoor/outdoor preference) from a free-form event description. Step 2 calls a weather API for the extracted location and date, and normalizes the response into structured weather data. Step 3 generates a recommendation message for the event organizer (proceed as planned / reschedule / move indoors) that combines both the event details and the weather data.

**Input:** `event_description_text`

**Output:** the structured event details from Step 1, the structured weather data from Step 2, and the recommendation message from Step 3

**Step 1 — Template:** `PromptTemplate` | **Parser:** `PydanticOutputParser` (fields: `event_type`, `event_date`, `location`, `indoor_outdoor_preference`)

**Step 2 — Template:** `PromptTemplate` (used to format the weather API call/query from Step 1's fields; the API response itself is normalized into structured output, not freely generated by the LLM) | **Parser:** `PydanticOutputParser` (fields: `temperature`, `precipitation_chance`, `conditions`)

**Step 3 — Template:** `ChatPromptTemplate` (system persona: "event planning assistant") | **Parser:** `StrOutputParser`


In [72]:
event_description_text = """
We are organizing an outdoor college technology festival in Mumbai, India,
on October 26, 2026. The event will include student project demonstrations,
technical games, and food stalls. We prefer to hold the entire event outdoors,
but we can move activities to a covered indoor hall if the weather is unsuitable.
"""

In [73]:
class EventDetails(BaseModel):
    event_type: str = Field(description="The type of event")
    event_date: str = Field(description="The planned event date in YYYY-MM-DD format")
    location: str = Field(description="The event location")
    indoor_outdoor_preference: str = Field(description="Whether the event is planned indoors or outdoors")

In [74]:
pydantic_parser = PydanticOutputParser(pydantic_object=EventDetails)

In [75]:
input_temp = PromptTemplate.from_template(
    template="""Extract the event type, planned date, location, and indoor/outdoor
    preference from the following event description.

    Event Description: {event_description_text}

    Return the result in the required JSON format.
    {JSON_FORMATTING}"""
).partial(JSON_FORMATTING=pydantic_parser.get_format_instructions())

In [76]:
input = input_temp.invoke({"event_description_text": event_description_text})

response = llm.invoke(input)

event_data = pydantic_parser.parse(response.content)

print(event_data)

event_type='college technology festival' event_date='2026-10-26' location='Mumbai, India' indoor_outdoor_preference='outdoor, with fallback to indoor if weather is unsuitable'


In [78]:
class WeatherData(BaseModel):
    temperature: float = Field(
        description="Mean daily temperature in Celsius"
    )
    precipitation_chance: int = Field(
        description="Maximum daily precipitation probability in percent"
    )
    conditions: str = Field(
        description="Weather conditions for the event date"
    )

In [79]:
weather_parser = PydanticOutputParser(pydantic_object=WeatherData)

In [80]:
weather_temp = PromptTemplate.from_template(
    template="Fetch the weather forecast for {location} on {event_date}."
)

weather_query = weather_temp.invoke({
    "location": event_data.location,
    "event_date": event_data.event_date
})

print(weather_query.text)

Fetch the weather forecast for Mumbai, India on 2026-10-26.


In [81]:
import requests
import json

location_response = requests.get(
    "https://geocoding-api.open-meteo.com/v1/search",
    params={
        "name": event_data.location.split(",")[0],
        "count": 1,
        "language": "en",
        "format": "json"
    },
    timeout=20
)

location_response.raise_for_status()
location_data = location_response.json()

if not location_data.get("results"):
    raise ValueError("Location not found by the weather API.")

latitude = location_data["results"][0]["latitude"]
longitude = location_data["results"][0]["longitude"]

weather_response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_mean,precipitation_probability_max,weather_code",
        "timezone": "Asia/Kolkata",
        "start_date": event_data.event_date,
        "end_date": event_data.event_date
    },
    timeout=20
)

weather_response.raise_for_status()
weather_api_data = weather_response.json()

if not weather_api_data.get("daily", {}).get("time"):
    raise ValueError("No forecast data available for the event date.")

print(weather_api_data)

{'latitude': 19.086115, 'longitude': 72.85291, 'generationtime_ms': 0.08511543273925781, 'utc_offset_seconds': 19800, 'timezone': 'Asia/Kolkata', 'timezone_abbreviation': 'GMT+5:30', 'elevation': 8.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_probability_max': '%', 'weather_code': 'wmo code'}, 'daily': {'time': ['2026-10-26'], 'temperature_2m_mean': [30.9], 'precipitation_probability_max': [0], 'weather_code': [0]}}


In [82]:
weather_code = weather_api_data["daily"]["weather_code"][0]

weather_conditions = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    80: "Rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    95: "Thunderstorm",
    96: "Thunderstorm with hail",
    99: "Thunderstorm with heavy hail"
}

weather_data = weather_parser.parse(json.dumps({
    "temperature": weather_api_data["daily"]["temperature_2m_mean"][0],
    "precipitation_chance": weather_api_data["daily"]["precipitation_probability_max"][0],
    "conditions": weather_conditions.get(weather_code, f"Weather code {weather_code}")
}))

print(weather_data)

temperature=30.9 precipitation_chance=0 conditions='Clear sky'


In [83]:
recommendation_temp = ChatPromptTemplate([
    ("system", "You are an event planning assistant. Provide practical event recommendations based on the event details and actual weather data."),
    ("user", """Recommend whether to proceed as planned, reschedule, or move indoors.

    Event Details:
    {event_details}

    Weather Data:
    {weather_data}

    Consider temperature, precipitation probability, weather conditions,
    and the indoor/outdoor preference. Explain the recommendation briefly.
    Do not invent weather information.""")
])

In [85]:
input = recommendation_temp.invoke({
    "event_details": event_data.model_dump_json(),
    "weather_data": weather_data.model_dump_json()
})

response = llm.invoke(input)

str_parser = StrOutputParser()

recommendation = str_parser.parse(response.content)

print(recommendation)

**Recommendation:** Proceed with the festival as planned (outdoor), with no need to move indoors or reschedule.

**Why:**  
- **Temperature:** 30.9 °C is warm but typical for Mumbai in late October; it’s manageable with shade, fans, and ample drinking water.  
- **Precipitation:** 0 % chance of rain – no risk of wet conditions.  
- **Conditions:** Clear sky – ideal for outdoor activities.  

**Practical tip:** Set up sun‑shades or tents, provide hydration stations, and consider light, breathable dress code to keep participants comfortable.


---

## Exercise 5: Customer Support Chat Log to Escalation Report Pipeline

### Step 1: Extract Structured Information from the Chat Log

Use `PromptTemplate` and `PydanticOutputParser` to extract the customer's issue, tone, and resolution attempts from a support chat log.

In [86]:
support_chat_log = """
Customer: I have been trying to access my account for three days, but
the login page keeps showing an error. This is extremely frustrating.

Support Agent: I understand your frustration. Have you tried resetting
your password?

Customer: Yes, I reset my password twice, cleared my browser cache,
and tried using another browser, but nothing worked.

Support Agent: I apologize for the inconvenience. We may need to
investigate this issue further.
"""

In [87]:
class SupportChatDetails(BaseModel):
    issue: str = Field(
        description="The main problem reported by the customer"
    )
    customer_tone: str = Field(
        description="The customer's emotional tone"
    )
    resolution_attempted: list[str] = Field(
        description="Troubleshooting steps already attempted"
    )


support_chat_parser = PydanticOutputParser(
    pydantic_object=SupportChatDetails
)

In [88]:
support_chat_prompt = PromptTemplate.from_template(
    """
    Extract structured information from the customer support chat log.

    Identify:
    1. The customer's main issue.
    2. The customer's emotional tone.
    3. All troubleshooting or resolution steps already attempted.

    Do not invent information that is not present in the chat log.

    Chat Log:
    {chat_log}

    {format_instructions}
    """
)

In [89]:
support_chat_input = support_chat_prompt.invoke({
    "chat_log": support_chat_log,
    "format_instructions": support_chat_parser.get_format_instructions()
})

support_chat_response = llm.invoke(support_chat_input)

support_chat_details = support_chat_parser.parse(
    support_chat_response.content
)

print(support_chat_details)

issue='Unable to access the account because the login page shows an error.' customer_tone='Frustrated' resolution_attempted=['Reset password twice', 'Cleared browser cache', 'Tried using a different browser']


In [90]:
class EscalationReadiness(BaseModel):
    missing_fields: list[str] = Field(
        description="Information missing that would help investigate or resolve the issue"
    )
    is_ready_to_escalate: bool = Field(
        description="Whether the issue has enough information to be escalated for further investigation"
    )


readiness_parser = PydanticOutputParser(
    pydantic_object=EscalationReadiness
)

In [91]:
readiness_prompt = PromptTemplate.from_template(
    """
    Evaluate whether this customer support case is ready for escalation.

    Consider the reported issue, customer tone, troubleshooting attempts,
    and whether additional information is needed for investigation.

    Support Case:
    {support_details}

    Identify any important missing information.
    Set is_ready_to_escalate to true if the issue can reasonably be
    escalated for further investigation, even if some details are missing.

    Do not invent facts.

    {format_instructions}
    """
)

In [92]:
readiness_input = readiness_prompt.invoke({
    "support_details": support_chat_details.model_dump_json(),
    "format_instructions": readiness_parser.get_format_instructions()
})

readiness_response = llm.invoke(readiness_input)

readiness_result = readiness_parser.parse(
    readiness_response.content
)

print(readiness_result)

missing_fields=['Exact error message or error code displayed on the login page', 'Account identifier (e.g., email address or username) being used', 'Date and time when the issue was first observed', 'Device type and operating system details', 'Network environment (e.g., corporate VPN, public Wi‑Fi)', 'Any recent changes to the account (e.g., password change, security settings)'] is_ready_to_escalate=True


In [93]:
escalation_report_prompt = ChatPromptTemplate([
    (
        "system",
        "You are a customer support escalation specialist. "
        "Create a clear, professional escalation report using only "
        "the information provided. Do not invent customer details."
    ),
    (
        "user",
        """
        Generate a customer support escalation report using the information below.

        Support Case Details:
        {support_details}

        Escalation Readiness:
        {readiness_result}

        Include:
        1. Customer Issue
        2. Customer Tone
        3. Troubleshooting Already Attempted
        4. Missing Information
        5. Escalation Status
        6. Recommended Next Steps

        Keep the report concise, clear, and professional.
        """
    )
])

In [94]:
escalation_input = escalation_report_prompt.invoke({
    "support_details": support_chat_details.model_dump_json(),
    "readiness_result": readiness_result.model_dump_json()
})

escalation_response = llm.invoke(escalation_input)

str_parser = StrOutputParser()

escalation_report = str_parser.parse(
    escalation_response.content
)

print(escalation_report)

**Customer Support Escalation Report**

---

### 1. Customer Issue  
- Unable to access the account; the login page displays an error.

### 2. Customer Tone  
- Frustrated

### 3. Troubleshooting Already Attempted  
- Reset password twice  
- Cleared browser cache  
- Tried using a different browser  

### 4. Missing Information  
The following details are required to proceed with a thorough investigation:  

1. Exact error message or error code shown on the login page  
2. Account identifier (e.g., email address or username) being used  
3. Date and time when the issue was first observed  
4. Device type and operating system details  
5. Network environment (e.g., corporate VPN, public Wi‑Fi)  
6. Any recent changes to the account (e.g., password change, security settings)

### 5. Escalation Status  
- **Ready to Escalate:** Yes (all initial troubleshooting steps have been performed)

### 6. Recommended Next Steps  
1. **Gather Missing Information** – Contact the customer promptly to 

---

## Exercise 6: Workout Goal to Weekly Plan Pipeline

### Step 1: Extract Structured Workout Requirements

Use `PromptTemplate` and `PydanticOutputParser` to extract the user's fitness goal, fitness level, available workout days, and equipment access from a natural-language request.

In [95]:
workout_request = """
I want to build muscle and improve my overall strength.
I am a beginner and can work out four days per week.
I have access to dumbbells and resistance bands at home,
but I do not have access to a gym.
"""

In [96]:
class WorkoutRequirements(BaseModel):
    goal_type: str = Field(
        description="The user's primary fitness goal"
    )
    fitness_level: str = Field(
        description="The user's current fitness experience level"
    )
    days_available: int = Field(
        description="Number of days per week available for workouts"
    )
    equipment_access: list[str] = Field(
        description="Equipment available to the user for workouts"
    )


workout_parser = PydanticOutputParser(
    pydantic_object=WorkoutRequirements
)

In [97]:
workout_prompt = PromptTemplate.from_template(
    """
    Extract the user's workout requirements from the request below.

    Identify:
    1. Primary fitness goal.
    2. Current fitness level.
    3. Number of available workout days per week.
    4. Available workout equipment.

    Do not invent information that is not provided.

    User Request:
    {workout_request}

    {format_instructions}
    """
)

In [98]:
workout_input = workout_prompt.invoke({
    "workout_request": workout_request,
    "format_instructions": workout_parser.get_format_instructions()
})

workout_response = llm.invoke(workout_input)

workout_requirements = workout_parser.parse(
    workout_response.content
)

print(workout_requirements)

goal_type='build muscle and improve overall strength' fitness_level='beginner' days_available=4 equipment_access=['dumbbells', 'resistance bands']


In [99]:
class WorkoutDay(BaseModel):
    day: str = Field(
        description="Day of the week for the workout or rest day"
    )
    focus: str = Field(
        description="Primary workout focus for the day"
    )
    exercises: list[str] = Field(
        description="Exercises planned for the day"
    )


class WeeklyWorkoutPlan(BaseModel):
    weekly_plan: list[WorkoutDay] = Field(
        description="Day-by-day workout schedule for the week"
    )


weekly_plan_parser = PydanticOutputParser(
    pydantic_object=WeeklyWorkoutPlan
)

In [100]:
weekly_plan_prompt = ChatPromptTemplate([
    (
        "system",
        "You are a fitness planning assistant. Create a practical, "
        "beginner-friendly weekly workout plan based on the user's "
        "goals and available equipment. Include recovery days and "
        "avoid scheduling more workout days than the user requested. "
        "Keep exercises appropriate for home workouts."
    ),
    (
        "human",
        """
        Create a structured seven-day weekly workout plan using these requirements.

        Workout Requirements:
        {workout_requirements}

        The plan must:
        1. Cover all seven days of the week.
        2. Schedule workouts on exactly the number of days specified.
        3. Include a focus and exercise list for every day.
        4. Include suitable rest or recovery days.
        5. Use only the equipment available to the user.
        6. Be appropriate for the user's fitness level.

        {format_instructions}
        """
    )
])

In [105]:
print("=" * 60)
print("          WEEKLY WORKOUT PLAN")
print("=" * 60)

for workout_day in weekly_workout_plan.weekly_plan:
    print(f"\nDay       : {workout_day.day}")
    print(f"Focus     : {workout_day.focus}")
    print("Exercises :")

    for index, exercise in enumerate(workout_day.exercises, start=1):
        print(f"  {index}. {exercise}")

    print("-" * 60)

          WEEKLY WORKOUT PLAN

Day       : Monday
Focus     : Upper Body Strength
Exercises :
  1. Dumbbell Chest Press (floor)
  2. Dumbbell Bent‑Over Row
  3. Resistance Band Pull‑Apart
  4. Dumbbell Shoulder Press
  5. Resistance Band Triceps Extension
------------------------------------------------------------

Day       : Tuesday
Focus     : Rest & Recovery
Exercises :
  1. Gentle stretching and mobility work
------------------------------------------------------------

Day       : Wednesday
Focus     : Lower Body Strength
Exercises :
  1. Goblet Squat (dumbbell)
  2. Resistance Band Romanian Deadlift
  3. Dumbbell Walking Lunge
  4. Resistance Band Lateral Walks (band around thighs)
  5. Dumbbell Calf Raise
------------------------------------------------------------

Day       : Thursday
Focus     : Rest & Recovery
Exercises :
  1. Gentle stretching and mobility work
------------------------------------------------------------

Day       : Friday
Focus     : Full Body Condition